# 00 — Colab launcher

Bootstrap pour lancer le pipeline RLHF sur Google Colab. Exécutez les cellules dans l'ordre, puis ouvrez les trois notebooks numérotés dans le panneau de fichiers à gauche.

**Prérequis** : `Runtime → Change runtime type → T4 GPU` (ou mieux).

## 1. Vérifier le GPU

Doit afficher `Tesla T4` (ou L4 / A100). Si vous voyez `No GPU detected`, retournez dans *Runtime → Change runtime type*.

In [ ]:
!nvidia-smi || echo 'No GPU detected'

## 2. Cloner le repo sur la bonne branche

In [ ]:
%cd /content
!rm -rf repo
!git clone --branch feat/Reinforcement_Learning_from_Human_Feedback \
    https://github.com/Crams0n/Project-Ethical-Alignment-of-Small-Language-Models.git repo
%cd /content/repo
!git log -1 --oneline

## 3. Installer les dépendances

~3 minutes la première fois (bitsandbytes pèse 60 Mo, torch est déjà préinstallé sur Colab).

In [ ]:
!pip install -q -r /content/repo/requirements.txt

## 4. Monter Google Drive (recommandé)

Persistance des checkpoints même si la session Colab est interrompue. Les artefacts iront dans `MyDrive/rlhf_outputs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p '/content/drive/MyDrive/rlhf_outputs'
!rm -rf /content/repo/outputs
!ln -sf '/content/drive/MyDrive/rlhf_outputs' /content/repo/outputs
!ls -la /content/repo/outputs

## 5. (Optionnel) Login Hugging Face

Qwen2.5 et HH-RLHF sont publics, mais un token évite les rate-limits sur de longs runs. Créez-en un sur https://huggingface.co/settings/tokens (read-only suffit).

In [ ]:
from huggingface_hub import login
login()  # collez votre token quand demandé

## 6. Sanity check

Doit afficher `device=cuda` et tous les imports OK.

In [ ]:
import sys; sys.path.insert(0, '/content/repo')
import os; os.chdir('/content/repo')

import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

from src.utils.device import runtime_profile
from src.utils.config import load_config
from trl import RewardTrainer, RLOOTrainer

profile = runtime_profile()
cfg = load_config('configs/config.yaml')
print('base model:', cfg['base_model'])
print('OK — prêt à lancer les notebooks 01 → 02 → 03')

## 7. Suite

Dans le panneau de fichiers à gauche (`/content/repo/notebooks/`), ouvrez dans l'ordre :

1. **`01_reward_model_training.ipynb`** — entraîne le reward model (~45-60 min sur T4).
2. **`02_rloo_training.ipynb`** — RLOO contre ce reward model (~1.5-3 h).
3. **`03_evaluation_ethics.ipynb`** — baseline vs aligné sur ETHICS (~15-30 min).

Chaque notebook écrit dans `outputs/` (symlinké vers Drive), donc le suivant retrouve automatiquement les artefacts.

### Conseil — smoke test d'abord

Avant de lancer 3-5 h de calcul, éditez `configs/config.yaml` pour valider le pipeline rapidement :
```yaml
reward_model:
  num_train_samples: 200
  num_eval_samples: 50
rloo:
  num_prompts: 100
evaluation:
  samples_per_subset: 20
```
Tout le pipeline tourne alors en ~20 min. Si OK, restaurez les vraies valeurs et relancez.